# 📦 Notebook B1 — Path B: Data Prep & Per-Token Language Tagging

**Path B Concrete Specs — Stage 1: Data Preparation**

**Path B requires per-token language labels** to train PolyWhisper's Mixture-of-LoRA-Experts (MoLE) router.

### What this notebook produces:
1. **Per-token language tags** ($0 = \text{Bengali}$, $1 = \text{English}$, $-100 = \text{ignore}$) via automated Unicode script range detection (`0x0980-0x09FF` vs `0x0041-0x007A`).
2. **Bengali Expert Subset** (`train_bengali_expert.csv`): Monolingual Bengali utterances to train the Bengali LoRA expert in **Notebook B2**.
3. **English Expert Subset** (`train_english_expert.csv`): English-dense utterances to train the English LoRA expert in **Notebook B3**.
4. **Router Training & Val Sets** (`train_router_tagged.json`, `val_router_tagged.json`): Code-switched utterances with pre-aligned token IDs, language labels, and audio timestamp boundaries (`start_sec`, `end_sec`) for the Router in **Notebook B4**.

> 📁 **Drive Storage**: Connected to main folder: [hoichoi on Google Drive](https://drive.google.com/drive/folders/1zqBtJvJ8ZwZNNpLCiS2F-x-yBpMxYXfE?usp=sharing). Reuses cached audio in `mucs_raw/` if already present.

---
⏭️ **Next steps:** Open **Notebook B2** (Bengali Expert) and **Notebook B3** (English Expert) in two separate tabs/GPUs and train them in parallel.

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 1 — Mount Drive + Set Paths
# Main Google Drive Folder: https://drive.google.com/drive/folders/1zqBtJvJ8ZwZNNpLCiS2F-x-yBpMxYXfE?usp=sharing
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
from google.colab import drive
drive.mount('/content/drive')

import os

DRIVE_ROOT   = '/content/drive/MyDrive/hoichoi'
DRIVE_AUDIO  = DRIVE_ROOT + '/mucs_raw'       # Raw audio directory
DRIVE_DATA_B = DRIVE_ROOT + '/data_path_b'   # Path B datasets

os.makedirs(DRIVE_AUDIO,  exist_ok=True)
os.makedirs(DRIVE_DATA_B, exist_ok=True)

print('✅ Drive mounted successfully')
print('   Main folder →', DRIVE_ROOT)
print('   Audio root  →', DRIVE_AUDIO)
print('   Path B data →', DRIVE_DATA_B)

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 2 — Install Required Dependencies
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import subprocess, sys

print('Installing dependencies (transformers, pandas, scikit-learn, tqdm) ...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'transformers>=4.40.0', 'pandas', 'scikit-learn', 'tqdm'],
               check=True)
print('✅ Dependencies installed successfully')

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 3 — Download / Verify MUCS Raw Data
# Reuses cached audio on Drive if already present
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import os, tarfile, urllib.request
from pathlib import Path

URL      = 'https://www.openslr.org/resources/104/Bengali-English_train.tar.gz'
TAR_PATH = DRIVE_AUDIO + '/Bengali-English_train.tar.gz'
DATA_DIR = Path(DRIVE_AUDIO)

# Check if already extracted (check for text, segments, and wav files)
text_files = list(DATA_DIR.rglob('text'))
seg_files  = list(DATA_DIR.rglob('segments'))
wav_files  = list(DATA_DIR.rglob('*.wav'))

if text_files and seg_files and len(wav_files) > 10:
    print(f'✅ MUCS data already present on Drive ({len(wav_files)} master recordings found).')
    print('   Skipping download — reusing existing audio files!')
else:
    if not os.path.exists(TAR_PATH):
        print('Downloading MUCS Bengali-English dataset (~3.9 GB from OpenSLR) → Drive ...')
        print('(This takes ~5-10 minutes. Please keep the Colab tab active.)')
        def _prog(count, block, total):
            mb  = count * block / 1e6
            tot = total / 1e6
            pct = min(100, int(mb * 100 / max(tot, 1)))
            print(f'\r  {pct}%  {mb:.0f} / {tot:.0f} MB', end='', flush=True)
        urllib.request.urlretrieve(URL, TAR_PATH, reporthook=_prog)
        print('\n✅ Download finished.')
    else:
        print('Archive found on Drive — extracting ...')

    print('Extracting tar archive ...')
    with tarfile.open(TAR_PATH, 'r:gz') as tf:
        tf.extractall(DATA_DIR)
    print('✅ Extracted to', DATA_DIR)

text_files = list(DATA_DIR.rglob('text'))
seg_files  = list(DATA_DIR.rglob('segments'))
if not text_files or not seg_files:
    raise RuntimeError('❌ transcripts/text or transcripts/segments missing after extract. Check archive contents.')

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 4 — Kaldi Parse, Script Tagging & Split Generation
# Bengali Unicode (0x0980-0x09FF) vs Latin (0x0041-0x007A)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import json, unicodedata
from pathlib import Path
import pandas as pd
import numpy as np
from tqdm import tqdm
from transformers import WhisperTokenizer
from sklearn.model_selection import train_test_split

SEED = 42
np.random.seed(SEED)

# Unicode ranges
BENGALI_RANGE = (0x0980, 0x09FF)
LATIN_RANGE   = (0x0041, 0x007A)

def char_script(c: str) -> str:
    cp = ord(c)
    if BENGALI_RANGE[0] <= cp <= BENGALI_RANGE[1]: return 'b'
    if LATIN_RANGE[0]   <= cp <= LATIN_RANGE[1]:   return 'l'
    return ''

def classify_token_lang(tok_text: str, fallback_lang: int = 0) -> int:
    """
    Maps decoded subword token to language ID:
      0 = Bengali (contains Bengali Unicode characters)
      1 = English (contains Latin Unicode characters)
      fallback_lang = punctuation/whitespace/digits (carries previous context)
    """
    has_b = any(BENGALI_RANGE[0] <= ord(c) <= BENGALI_RANGE[1] for c in tok_text)
    has_l = any(LATIN_RANGE[0]   <= ord(c.lower()) <= LATIN_RANGE[1] for c in tok_text)
    if has_b and not has_l: return 0
    if has_l and not has_b: return 1
    if has_b and has_l:     return 0   # Transliterated / mixed default to Bengali
    return fallback_lang

# Load Whisper tokenizer for token ID alignment
print('Loading Whisper tokenizer (SayedShaun/bengali-whisper-medium) ...')
tokenizer = WhisperTokenizer.from_pretrained('SayedShaun/bengali-whisper-medium')

# Parse Kaldi format transcripts
DATA_DIR   = Path(DRIVE_AUDIO)
seg_files  = [p for p in DATA_DIR.rglob('segments') if 'test' not in str(p).lower() and p.is_file()]
text_files = [p for p in DATA_DIR.rglob('text')     if 'test' not in str(p).lower() and p.is_file()]
wav_map    = {p.stem: str(p) for p in DATA_DIR.rglob('*.wav')}

print(f'Audio recordings found on Drive : {len(wav_map)}')

# 1. Parse segments
seg_dict = {}
for sf in seg_files:
    for line in open(sf, encoding='utf-8'):
        parts = line.strip().split()
        if len(parts) >= 4:
            uid, rec_id, s_sec, e_sec = parts[0], parts[1], float(parts[2]), float(parts[3])
            seg_dict[uid] = (rec_id, s_sec, e_sec)
print(f'Total segments parsed          : {len(seg_dict)}')

# 2. Parse transcripts
raw_records = []
for tf in text_files:
    for line in open(tf, encoding='utf-8'):
        line = line.strip()
        if not line or line.startswith('#'): continue
        parts = line.split(None, 1)
        if len(parts) < 2: continue
        uid, raw_text = parts[0], parts[1]
        
        if uid not in seg_dict: continue
        rec_id, s_sec, e_sec = seg_dict[uid]
        apath = wav_map.get(rec_id)
        if not apath: continue
        
        trans = unicodedata.normalize('NFC', raw_text.strip())
        words = trans.split()
        b_words = sum(1 for w in words if any(char_script(c) == 'b' for c in w))
        l_words = sum(1 for w in words if any(char_script(c) == 'l' for c in w))
        
        raw_records.append({
            'utt_id': uid,
            'audio_path': apath,
            'start_sec': s_sec,
            'end_sec': e_sec,
            'duration': round(e_sec - s_sec, 2),
            'transcript': trans,
            'b_words': b_words,
            'l_words': l_words,
            'is_cs': (b_words > 0 and l_words > 0),
            'is_bengali_only': (b_words > 0 and l_words == 0),
            'is_english_dense': (l_words >= 3 or (l_words > 0 and b_words <= 2)),
        })

print(f'Total valid audio-transcript pairs: {len(raw_records)}')
b_mono        = [r for r in raw_records if r['is_bengali_only']]
en_dense      = [r for r in raw_records if r['is_english_dense']]
cs_utterances = [r for r in raw_records if r['is_cs']]
print(f'  Bengali-only utterances       : {len(b_mono)}')
print(f'  English-dense utterances     : {len(en_dense)}')
print(f'  Code-switched utterances (CS): {len(cs_utterances)}')

# ── 1. Create Monolingual Expert Datasets ───────────────────
# Bengali expert dataset (Stage 2)
df_bengali = pd.DataFrame(b_mono)
if len(df_bengali) > 2500:
    df_bengali = df_bengali.sample(n=2500, random_state=SEED)
bn_train, bn_val = train_test_split(df_bengali, test_size=0.1, random_state=SEED)
bn_train.to_csv(DRIVE_DATA_B + '/train_bengali_expert.csv', index=False, encoding='utf-8')
bn_val.to_csv(DRIVE_DATA_B + '/val_bengali_expert.csv', index=False, encoding='utf-8')
print(f'\n✅ Saved Bengali Expert data: {len(bn_train)} train, {len(bn_val)} val')

# English expert dataset (Stage 3)
df_english = pd.DataFrame(en_dense)
if len(df_english) > 2500:
    df_english = df_english.sample(n=2500, random_state=SEED)
en_train, en_val = train_test_split(df_english, test_size=0.1, random_state=SEED)
en_train.to_csv(DRIVE_DATA_B + '/train_english_expert.csv', index=False, encoding='utf-8')
en_val.to_csv(DRIVE_DATA_B + '/val_english_expert.csv', index=False, encoding='utf-8')
print(f'✅ Saved English Expert data: {len(en_train)} train, {len(en_val)} val')

# ── 2. Create Per-Token Tagged Code-Switched Dataset ─────────
print('\nGenerating per-token language tags for Router training (Stage 4) ...')
tagged_cs = []
for r in tqdm(cs_utterances, desc='Token-tagging CS data'):
    text = r['transcript']
    tok_ids = tokenizer.encode(text, add_special_tokens=False)
    
    tok_langs = []
    cur_lang  = 0
    for tid in tok_ids:
        tok_str = tokenizer.decode([tid])
        cur_lang = classify_token_lang(tok_str, fallback_lang=cur_lang)
        tok_langs.append(cur_lang)
    
    tagged_cs.append({
        'utt_id': r['utt_id'],
        'audio_path': r['audio_path'],
        'start_sec': r['start_sec'],
        'end_sec': r['end_sec'],
        'duration': r['duration'],
        'transcript': text,
        'token_ids': tok_ids,
        'token_langs': tok_langs,
        'n_bn_tokens': sum(1 for l in tok_langs if l == 0),
        'n_en_tokens': sum(1 for l in tok_langs if l == 1),
    })

# Train/Val split for router
train_router, val_router = train_test_split(tagged_cs, test_size=0.1, random_state=SEED)

with open(DRIVE_DATA_B + '/train_router_tagged.json', 'w', encoding='utf-8') as f:
    json.dump(train_router, f, ensure_ascii=False, indent=2)
with open(DRIVE_DATA_B + '/val_router_tagged.json', 'w', encoding='utf-8') as f:
    json.dump(val_router, f, ensure_ascii=False, indent=2)

pd.DataFrame(train_router).to_csv(DRIVE_DATA_B + '/train_router.csv', index=False, encoding='utf-8')
pd.DataFrame(val_router).to_csv(DRIVE_DATA_B + '/val_router.csv', index=False, encoding='utf-8')

print(f'✅ Router data saved:')
print(f'   Train tagged utterances: {len(train_router)}')
print(f'   Val tagged utterances  : {len(val_router)}')

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 5 — Sanity Check & Token Statistics
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import os, json, pandas as pd

print('=' * 65)
print('  PATH B DATA PREPARATION SUMMARY')
print('=' * 65)

files_to_check = [
    ('Bengali Expert Train', 'train_bengali_expert.csv'),
    ('Bengali Expert Val',   'val_bengali_expert.csv'),
    ('English Expert Train', 'train_english_expert.csv'),
    ('English Expert Val',   'val_english_expert.csv'),
    ('Router Tagged Train',  'train_router_tagged.json'),
    ('Router Tagged Val',    'val_router_tagged.json'),
]

for label, fname in files_to_check:
    p = DRIVE_DATA_B + '/' + fname
    exists = os.path.exists(p)
    if exists:
        if fname.endswith('.csv'):
            count = len(pd.read_csv(p))
        else:
            count = len(json.load(open(p, encoding='utf-8')))
        print(f'  ✅ {label:<22} : {count:>5} records  ({fname})')
    else:
        print(f'  ❌ {label:<22} : MISSING ({fname})')

# Inspect a sample tagged code-switched utterance
with open(DRIVE_DATA_B + '/train_router_tagged.json', encoding='utf-8') as f:
    samples = json.load(f)

print('\n' + '=' * 65)
print('  SAMPLE PER-TOKEN LANGUAGE TAGGED UTTERANCE')
print('=' * 65)
sample = samples[0]
print(f'Utterance ID: {sample["utt_id"]}')
print(f'Audio Path  : {sample["audio_path"]}')
print(f'Slice Timing: {sample["start_sec"]} s to {sample["end_sec"]} s ({sample.get("duration")} s)')
print(f'Transcript  : {sample["transcript"]}\n')
print('Token Breakdown:')
from transformers import WhisperTokenizer
tokenizer = WhisperTokenizer.from_pretrained('SayedShaun/bengali-whisper-medium')

tagged_display = []
for tid, lang in zip(sample['token_ids'], sample['token_langs']):
    tok_decoded = repr(tokenizer.decode([tid]))[1:-1]
    lang_tag    = '[BN]' if lang == 0 else '[EN]'
    tagged_display.append(f'{lang_tag}{tok_decoded}')

print(' '.join(tagged_display))

total_bn_tok = sum(s['n_bn_tokens'] for s in samples)
total_en_tok = sum(s['n_en_tokens'] for s in samples)
total_tokens = total_bn_tok + total_en_tok
print('\nRouter Token Distribution:')
print(f'  Bengali Tokens : {total_bn_tok:>6} ({total_bn_tok*100/total_tokens:.1f}%)')
print(f'  English Tokens : {total_en_tok:>6} ({total_en_tok*100/total_tokens:.1f}%)')
print(f'  Total Tokens   : {total_tokens:>6}')

print('=' * 65)
print('✅ Notebook B1 complete!')
print('   👉 Next: Open Notebook B2 (Bengali Expert) and Notebook B3 (English Expert)')
print('      in separate Colab tabs to train in parallel.')
print('=' * 65)